# Nhóm 16 — P10-T05: Benchmark và profiling V5

Người thực hiện: **Lê Thượng Đế (23120232)**.

Đo V0–V5 cùng một phiên Tesla T4, một warm-up và 5 runs/cấu hình. Profiling đối chiếu S6 V3/V5; thời gian có profiler không tham gia bảng benchmark. Bằng chứng correctness từ P10-T02 chỉ được dùng sau khi kiểm tra source production khớp.

Chạy tuần tự các cell. Cell cuối cho phép lưu bằng chứng từng phần khi profiler thất bại; task chỉ hoàn tất khi có phân tích, screenshots và bàn giao Drive.


In [ ]:
# @title Cấu hình P10-T05
SOURCE_MODE = "github"  # Official runs verify production with Git.
GIT_URL = "https://github.com/nh-atuan/panorama-stitching.git"
GIT_REF = "ba97ed780785135e9d20ce36aac852efc4ebc898"  # @param {type:"string"}
SOURCE_ZIP = "/content/Group16_P10_Source.zip"  # @param {type:"string"}
DATASET_ZIP = "/content/dataset.zip"  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
# Optional: a parent folder, e.g. /content/drive/MyDrive/CSC14120_Group16_Results
DRIVE_RESULTS_DIR = ""  # @param {type:"string"}
RUN_PROFILING = True  # @param {type:"boolean"}
NCU_TIMEOUT_SECONDS = 600  # @param {type:"integer"}
# Chọn trước site2/large làm ca chính; small/large để khảo sát độ nhạy kích thước.
PROFILE_CASES = [("site2", "large"), ("site1", "small"), ("site1", "large"), ("site3", "large")]
NCU_KERNELS = {3: ["copy_a_kernel", "gather_b_kernel"], 4: ["render_canvas_kernel"]}
PRODUCTION_SHA = 'ab8df98d7f15bb42d5e3306cf03a38311248b4a8'  # Source đã nghiệm thu P10-T02; fetch/diff xác nhận bên dưới.


## 1. Runtime và source

Giữ workspace, source và binary cố định trong lượt đo. Notebook lưu argv, exit code, stdout và stderr cho từng lệnh. Thiếu profiler hoặc quyền đọc counter được ghi nhận là lỗi thu thập bằng chứng.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from google.colab import files
from IPython.display import Markdown, display, Image as DisplayImage
import csv
import hashlib
import io
import json
import math
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
WORK = Path(tempfile.mkdtemp(prefix="group16_p10_t05_", dir="/content"))
PACKAGE = WORK / "package"
# Reset evidence state when starting a new workspace in the same Python runtime.
BENCHMARK_OK = False
SOURCE = {}
BASELINE = {}
AVERAGES = {}
PARSED = {}
SPEEDUPS = []
PROFILE_RESULTS = []
VARIABILITY = []
BENCH = PACKAGE / "v5/benchmark"
PROFILE = PACKAGE / "v5/profiling"
for folder in (BENCH, PROFILE, PACKAGE / "logs", PACKAGE / "report"):
    folder.mkdir(parents=True, exist_ok=True)

def dump(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def upload_one(message):
    print(message)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one file.")
    return Path(next(iter(uploaded))).resolve()

def unpack(archive, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        for item in source.infolist():
            name = PurePosixPath(item.filename)
            if (name.is_absolute() or ".." in name.parts or chr(92) in item.filename
                    or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError(f"Unsafe archive entry: {item.filename}")
        source.extractall(target)
    return target

def run(args, label, timeout=600, required=True):
    args = list(map(str, args))
    prefix = PACKAGE / "logs" / label
    prefix.parent.mkdir(parents=True, exist_ok=True)
    print("+", " ".join(args), flush=True)
    try:
        result = subprocess.run(args, cwd=PROJECT if "PROJECT" in globals() else WORK,
                                capture_output=True, text=True, timeout=timeout)
        stdout, stderr, rc = result.stdout, result.stderr, result.returncode
    except subprocess.TimeoutExpired as error:
        stdout, stderr, rc = error.stdout or "", error.stderr or "", -1
        stdout = stdout.decode(errors="replace") if isinstance(stdout, bytes) else stdout
        stderr = stderr.decode(errors="replace") if isinstance(stderr, bytes) else stderr
        stderr += "\nTIMEOUT"
    except OSError as error:
        stdout, stderr, rc = "", str(error), -1
    prefix.with_suffix(".stdout").write_text(stdout, encoding="utf-8")
    prefix.with_suffix(".stderr").write_text(stderr, encoding="utf-8")
    record = {"command": args, "returncode": rc, "stdout": str(prefix.with_suffix(".stdout").relative_to(PACKAGE)),
              "stderr": str(prefix.with_suffix(".stderr").relative_to(PACKAGE))}
    dump(prefix.with_suffix(".command.json"), record)
    print(stdout[-4000:])
    if stderr:
        print(stderr[-2000:])
    if required and rc != 0:
        raise RuntimeError(f"{label}: exit {rc}; logs: {prefix}. Run the export cell to retain evidence.")
    return record, stdout, stderr

# Add profiler binaries already installed by the runtime to PATH.
for pattern in ("/opt/nvidia/nsight-systems/*/bin/nsys", "/opt/nvidia/nsight-compute/*/ncu",
                "/usr/local/cuda*/bin/ncu"):
    import glob
    for executable in sorted(glob.glob(pattern), reverse=True):
        if shutil.which(Path(executable).name) is None:
            os.environ["PATH"] = str(Path(executable).parent) + os.pathsep + os.environ["PATH"]
record, gpu, _ = run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], "gpu")
if not any(line.split(",")[0].strip() in {"Tesla T4", "NVIDIA T4", "T4"} for line in gpu.splitlines()):
    raise RuntimeError("Select a Tesla T4 runtime.")
run(["nvcc", "--version"], "nvcc")
for name in ("nsys", "ncu"):
    print(name, shutil.which(name) or "MISSING: profiling for this tool cannot pass")
print("Workspace:", WORK)


In [ ]:
# @title Lấy source đã tích hợp V5
if SOURCE_MODE == "github":
    PROJECT = WORK / "source"
    PROJECT.mkdir()
    run(["git", "init", "--quiet"], "git_init")
    run(["git", "fetch", "--depth", "1", "--", GIT_URL, GIT_REF], "git_fetch", timeout=180)
    run(["git", "checkout", "--detach", "--quiet", "FETCH_HEAD"], "git_checkout")
elif SOURCE_MODE == "zip":
    archive = Path(SOURCE_ZIP)
    if not archive.is_file():
        archive = upload_one("Upload source ZIP containing src/, scripts/ and tests/.")
    extracted = unpack(archive, WORK / "source_upload")
    roots = [p.parent.parent for p in extracted.rglob("src/main.cu")]
    if len(roots) != 1:
        raise ValueError("Expected exactly one src/main.cu in the source ZIP.")
    PROJECT = roots[0]
else:
    raise ValueError("Unsupported SOURCE_MODE")
for name in ("src/v5/v5_pipeline.cu", "src/v5/s6/v5_warp_blend.cu", "scripts/run_benchmark.py",
             "scripts/profiling/run_profiling.py", "scripts/verify/verify_v2_dataset.py"):
    if not (PROJECT / name).is_file():
        raise FileNotFoundError(name)
sys.path.insert(0, str(PROJECT / "scripts"))
os.chdir(PROJECT)
from verify.verify_v2_dataset import official_inputs, source_snapshot, parse_v2_csv
from run_benchmark import (parse_version_csv, summary_rows, write_summary_csv,
                           compute_speedups, write_speedup_csv, write_speedup_md)
from profiling.profiling_common import inspect_environment
SOURCE = source_snapshot()
dump(PACKAGE / "source_manifest.json", SOURCE)
dump(PACKAGE / "environment.json", inspect_environment(require_t4=True))
print("Source commit:", SOURCE.get("git_commit"))
print("Snapshot SHA-256:", SOURCE["snapshot_sha256"])

if SOURCE_MODE == "github":
    run(["git", "fetch", "--depth", "1", "--", GIT_URL, PRODUCTION_SHA], "production_fetch", timeout=180)
    run(["git", "diff", "--exit-code", PRODUCTION_SHA, "HEAD", "--", "src", "scripts/build.py"], "production_match")
else:
    raise ValueError("Use github mode to verify the P10-T02 production source before official measurements.")


## 2. Dataset cuối kỳ

Đối chiếu manifest và hash của 24 ảnh với bộ P7/P10-T02. Mọi phiên bản dùng cùng các ảnh này.


In [ ]:
# Prefer the exact final ZIP before trying to reproduce images from raw photos.
if not Path(DATASET_ZIP).is_file():
    try:
        from fetch_and_prepare_dataset import ensure_gdown
        ensure_gdown()
        import gdown
        download_dir = WORK / "official_dataset_download"
        download_dir.mkdir(exist_ok=True)
        gdown.download_folder(url="https://drive.google.com/drive/folders/1_J_q5q-K1g7ZNnczTWvXBzwpOsof9ghL?usp=sharing",
                              output=str(download_dir) + "/", use_cookies=False)
        archives = list(download_dir.rglob("dataset.zip"))
        if len(archives) == 1:
            DATASET_ZIP = str(archives[0])
    except Exception as error:
        print("Official ZIP unavailable, trying raw preparation:", error)

# @title Chuẩn bị và kiểm tra dataset
archive = Path(DATASET_ZIP)
if not archive.is_file():
    result, _, _ = run([sys.executable, "scripts/fetch_and_prepare_dataset.py", "--out", PROJECT / "dataset"],
                       "dataset_download", timeout=600, required=False)
    if result["returncode"] != 0:
        archive = upload_one("Download failed. Upload the official final dataset.zip with manifest.csv.")
if archive.is_file():
    extracted = unpack(archive, WORK / "dataset_upload")
    manifests = list(extracted.rglob("manifest.csv"))
    if len(manifests) != 1:
        raise ValueError("Dataset ZIP must have exactly one manifest.csv")
    DATASET = manifests[0].parent
else:
    DATASET = PROJECT / "dataset"
DATASET_CASES = official_inputs(DATASET)
MANIFEST_HASH = digest(DATASET / "manifest.csv")
shutil.copy2(DATASET / "manifest.csv", PACKAGE / "dataset_manifest.csv")
print("Validated cases:", len(DATASET_CASES))
print("Dataset manifest SHA-256:", MANIFEST_HASH)

if MANIFEST_HASH != "99fc712988075326a351d0bf580405e418bb28d24d56d4cc1b31505c480bdd9d":
    raise ValueError("Dataset manifest differs from accepted P7/P8 data")


In [ ]:
# @title Baseline cùng phiên
print("V0–V5 dùng cùng GPU, binary và dataset trong runtime này.")


## 3. Benchmark V0–V5

Một warm-up không tính giờ và 5 runs cho mỗi cấu hình. Báo trung bình, minimum, độ biến thiên, timing từng stage và truyền dữ liệu. Challenge được lưu riêng; ca thất bại không có speedup.


In [ ]:
# @title Build và chạy benchmark (không có profiler)
run([sys.executable, "scripts/build.py", "--compiler", "nvcc", "--target", "stitch", "--require-t4"],
    "build", timeout=600)
STITCH = PROJECT / "build/stitch"
RAW_BENCH = WORK / "benchmark_raw"
if RAW_BENCH.exists():
    raise RuntimeError("Benchmark output exists. Start a new workspace to avoid overwriting evidence.")
run([sys.executable, "scripts/run_benchmark.py", "--stitch", STITCH, "--dataset", DATASET,
     "--versions", "0", "1", "2", "3", "4", "--runs", "5", "--include-challenge", "--out", RAW_BENCH],
    "benchmark_v0_v1_v2_v3_v5", timeout=1800)
shutil.copytree(RAW_BENCH / "v5", BENCH, dirs_exist_ok=True)
for v in (0, 1, 2, 3):
    shutil.copytree(RAW_BENCH / f"v{v}", BENCH / f"reference_v{v}_same_session")
shutil.copy2(RAW_BENCH / "environment.json", BENCH / "environment.json")
AVERAGES = {v: {} for v in (0, 1, 2, 3, 4)}
PARSED = {}
for version in (0, 1, 2, 3, 4):
    for path in sorted((RAW_BENCH / f"v{version}").glob("*/*.csv")):
        if not path.read_text().strip():
            continue
        site, size = path.parent.name, path.stem
        text = path.read_text(encoding="utf-8")
        parsed = parse_version_csv(text, runs=5, version=version)
        if version == 2:
            parse_v2_csv(text, 5)
        AVERAGES[version][(site, size)] = parsed["average"]
        PARSED[(version, site, size)] = parsed
    for site in ("site1", "site2", "site3"):
        for size in ("small", "medium", "large"):
            if (site, size) not in AVERAGES[version]:
                raise ValueError(f"Missing standard benchmark V{version} {site}/{size}")
BASELINE = AVERAGES[0]
# Compare all versions measured on this runtime.
SPEEDUPS = [r for r in compute_speedups(AVERAGES, [0, 1, 2, 3, 4], include_challenge=True) if r["version"] == "V5"]
write_speedup_csv(BENCH / "speedup.csv", SPEEDUPS)
write_speedup_md(BENCH / "speedup.md", SPEEDUPS)
with (BENCH / "speedup.md").open("a", encoding="utf-8") as stream:
    stream.write("\nV5/V3 và V5/V0: cùng runtime T4; V0–V5 dùng cùng source và dataset.\n")
# Save all version comparisons, including the V1 reference.
all_speedups = compute_speedups(AVERAGES, [0, 1, 2, 3, 4], include_challenge=True)
write_speedup_csv(BENCH / "all_versions_speedup.csv", all_speedups)
write_speedup_md(BENCH / "all_versions_speedup.md", all_speedups)
with (BENCH / "v5_vs_all.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["site", "size", "V5_ms", "speedup_vs_v0", "speedup_vs_v1", "speedup_vs_v2", "speedup_vs_v3"])
    for key, current in AVERAGES[4].items():
        writer.writerow([*key, current["Total_ms"], *[
            AVERAGES[v][key]["Total_ms"] / current["Total_ms"] if key in AVERAGES[v] else ""
            for v in (0, 1, 2, 3)]])
write_summary_csv(BENCH / "summary.csv", [row for (v, site, size), parsed in PARSED.items() if v == 4 for row in summary_rows(site, size, parsed, version=v)])
BENCHMARK_OK = True
print("Benchmark CSVs validated; no profiler timing included.")


In [ ]:
# @title Biểu đồ, độ biến thiên và bảng speedup
run([sys.executable, "scripts/plot_benchmark.py", "--summary", BENCH / "summary.csv",
     "--speedup", BENCH / "speedup.csv", "--out", BENCH / "analysis"], "plots")
display(Markdown((BENCH / "speedup.md").read_text()))
VARIABILITY = []
for (version, site, size), parsed in PARSED.items():
    samples = [r["Total_ms"] for r in parsed["runs"]]
    mean = sum(samples) / len(samples)
    sd = math.sqrt(sum((value - mean)**2 for value in samples) / len(samples))
    VARIABILITY.append({"version": version, "site": site, "size": size,
                        "runs_ms": samples, "cv_pct": 100 * sd / mean})
dump(BENCH / "variability.json", VARIABILITY)
for row in VARIABILITY:
    if row["version"] == 4:
        print(f"{row['site']}/{row['size']}: CV={row['cv_pct']:.1f}% runs={row['runs_ms']}")
# Save an explicit V5/V3 plot; existing speedup.png compares to same-session V0.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for axis, site in zip(axes, ("site1", "site2", "site3")):
    selected = [next(r for r in SPEEDUPS if r["site"] == site and r["size"] == size)
                for size in ("small", "medium", "large")]
    axis.bar([r["size"] for r in selected], [r["speedup_vs_prev"] for r in selected])
    axis.axhline(1, color="gray", linestyle="--")
    axis.set(title=site, ylabel="V5 speedup vs V3 (same session)")
fig.tight_layout()
fig.savefig(BENCH / "analysis/speedup_vs_v3.png", dpi=150)
plt.close(fig)
for name in ("stage_share.png", "speedup_vs_v3.png", "speedup.png"):
    if name != "speedup.png" or BASELINE:
        display(DisplayImage(filename=str(BENCH / "analysis" / name)))

fig, ax = plt.subplots(figsize=(12, 4))
keys = [(site, size) for site in ("site1", "site2", "site3") for size in ("small", "medium", "large")]
xs = list(range(len(keys)))
ax.bar([x-.2 for x in xs], [AVERAGES[3][k]["S6_ms"] for k in keys], width=.4, label="V3")
ax.bar([x+.2 for x in xs], [AVERAGES[4][k]["S6_ms"] for k in keys], width=.4, label="V5")
ax.set_xticks(xs, ["/".join(k) for k in keys], rotation=35, ha="right")
ax.set_ylabel("S6 (ms)"); ax.legend(); fig.tight_layout()
fig.savefig(BENCH / "analysis/s6_v3_v5.png", dpi=150); plt.close(fig)


## 4. Profiling S6 V3/V5

NSYS thu timeline toàn pipeline và thao tác memset/copy. NCU thu riêng copy A và gather B của V3, renderer fusion của V5. Ưu tiên DRAM read/write bytes, L1/L2 traffic, duration, occupancy và registers. NCU một launch đầu có thể thuộc warm-up. Tổng counter hai kernel V3 chưa bao gồm memset; ghi riêng giới hạn đó và dùng NSYS cho toàn S6. Không dùng timing profiler làm speedup chính thức.


In [ ]:
# @title Thu NSYS/NCU — tiếp tục lưu kết quả nếu một profiler lỗi
PROFILE_RESULTS = []
run(["ncu", "--query-metrics"], "ncu_supported_metrics", required=False)
for site, size in PROFILE_CASES:
    if (site, size) not in AVERAGES[4]:
        raise ValueError(f"Profile case has no successful V5 benchmark: {site}/{size}")
if RUN_PROFILING:
    for version in (3, 4):
        for site, size in PROFILE_CASES:
            folder = PROFILE / f"v{version}" / site / size
            folder.mkdir(parents=True, exist_ok=True)
            nsys_out = folder / "nsys"
            record, out, err = run(
                [sys.executable, "scripts/profiling/run_profiling.py", "--stitch", STITCH, "--dataset", DATASET,
                 "--versions", str(version), "--sites", site, "--sizes", size, "--profilers", "nsys",
                 "--out", nsys_out, "--require-t4"], f"nsys_v{version}_{site}_{size}", timeout=900, required=False)
            summary_path = nsys_out / "summary.json"
            summary = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
            reports = list(nsys_out.rglob("*.nsys-rep"))
            success = (record["returncode"] == 0 and bool(reports) and bool(summary.get("profiling"))
                       and all(r["status"] == "ok" for r in summary["profiling"]))
            PROFILE_RESULTS.append({"version": version, "tool": "nsys", "site": site, "size": size, "ok": success,
                                    "log": record, "details": summary})
            for kernel in NCU_KERNELS[version]:
                target = folder / "ncu" / kernel
                target.mkdir(parents=True, exist_ok=True)
                report_base = target / "capture"
                command = ["ncu", "--set", "full", "--metrics", "dram__bytes_read.sum,dram__bytes_write.sum", "--target-processes", "all", "--kernel-name", "regex:" + kernel,
                           "--launch-count", "1", "--export", report_base,
                           STITCH, DATASET / site / f"{size}_left.jpg", DATASET / site / f"{size}_right.jpg",
                           target / "panorama.png", "--version", str(version), "--runs", "1", "--debug"]
                label = f"ncu_v{version}_{site}_{size}_{kernel}"
                record, out, err = run(command, label, timeout=NCU_TIMEOUT_SECONDS, required=False)
                report = report_base.with_suffix(".ncu-rep")
                success = (record["returncode"] == 0 and report.is_file()
                           and "ERR_NVGPUCTRPERM" not in out + err
                           and "No kernels were profiled" not in out + err)
                export_record = None
                if success:
                    export_record, metrics, export_error = run(
                        ["ncu", "--import", report, "--csv"], label + "_export", required=False)
                    (target / "metrics.csv").write_text(metrics, encoding="utf-8")
                    success = export_record["returncode"] == 0 and bool(metrics.strip())
                    raw_record, raw_metrics, _ = run(
                        ["ncu", "--import", report, "--page", "raw", "--csv"], label + "_raw_export", required=False)
                    (target / "raw_metrics.csv").write_text(raw_metrics, encoding="utf-8")
                    success = success and raw_record["returncode"] == 0 and bool(raw_metrics.strip())
                PROFILE_RESULTS.append({"version": version, "tool": "ncu", "site": site, "size": size, "kernel": kernel,
                                        "ok": success, "log": record, "export": export_record})

else:
    print("Profiling skipped. P10-T05 remains incomplete.")
dump(PROFILE / "profile_status.json", PROFILE_RESULTS)
for row in PROFILE_RESULTS:
    print(row["tool"], row["site"], row["size"], row.get("kernel", ""), "OK" if row["ok"] else "MISSING/FAILED")


## 5. Đọc bằng chứng

So NSYS trước/sau fusion: kernel count, memset, copy và tỷ trọng S6. So NCU theo cùng canvas: tổng read/write bytes copy A + gather B (V3) với renderer (V5), ghi rõ memset ngoài tổng counter. Kiểm occupancy/registers/cache/stalls để giải thích cả regression. Xuất ảnh từ report gốc và giữ đường dẫn truy xuất số đo.


In [ ]:
# @title Tùy chọn: thêm ảnh chụp Nsight (có thể chạy lại sau)
UPLOAD_SCREENSHOTS = False  # @param {type:"boolean"}
if UPLOAD_SCREENSHOTS:
    archive = upload_one("Upload a ZIP of NSYS timeline and NCU metric screenshots (PNG/JPG).")
    unpack(archive, PROFILE / "screenshots")
else:
    print("Screenshots pending: open traces in Nsight and add screenshots before completing P10-T05.")


In [ ]:
# @title Kiểm tra độ ổn định V3/V5 tại ca chính (không bật profiler)
STABILITY = []
site, size = 'site2', 'large'
for block_id in range(5):
    order = (3, 4) if block_id % 2 == 0 else (4, 3)
    records = {}
    for version in order:
        label = f'stability_block{block_id}_v{version}'
        record, text, error = run([STITCH, DATASET / site / f'{size}_left.jpg',
                                  DATASET / site / f'{size}_right.jpg',
                                  WORK / f'{label}_panorama.jpg', '--version', str(version),
                                  '--runs', '5', '--debug'], label, timeout=180)
        records[version] = parse_version_csv(text, runs=5, version=version)
    a, b = records[3]['average'], records[4]['average']
    STABILITY.append({'block': block_id, 'order': list(order), 'v3': records[3], 'v5': records[4],
                      'total_speedup': a['Total_ms'] / b['Total_ms'],
                      's6_speedup': a['S6_ms'] / b['S6_ms']})
dump(BENCH / 'paired_stability_site2_large.json', STABILITY)
import statistics
ratios = [row['total_speedup'] for row in STABILITY]
print('site2/large: V3/V5 paired-block speedup median/min/max =',
      statistics.median(ratios), min(ratios), max(ratios))
print('Blocks faster with V5:', sum(r > 1 for r in ratios), '/ 5')


In [ ]:
# @title Sinh bản nháp APOD từ số đo và checklist bàn giao
if not BENCHMARK_OK:
    raise RuntimeError("Benchmark is incomplete. Run the export cell to save partial evidence.")
lines = ["# P10-T05 — Benchmark và profiling V5", "",
         "Người thực hiện: Lê Thượng Đế (23120232). BẢN NHÁP: cần bổ sung phân tích profiling.", "",
         "## 1. Bottleneck", "",
         "Dữ liệu V3/V5 dưới đây được đo trong cùng runtime T4. V0 là baseline đo cùng phiên T4.", "",
         "## 2. Idea", "",
         "V5 gộp xóa nền, copy A và gather B thành một renderer S6; tái sử dụng S1–S4 V1 và S5 V3.", "",
         "## 3. Implementation", "", "Đối chiếu `src/v5/s6/`, `src/v5/v5_pipeline.cu` và source_manifest.json.", "",
         "## 4. Result", "", (BENCH / "speedup.md").read_text(), "",
         "| Site | Size | S6 V3 ms | S6 V5 ms | S6 speedup | S6 share V5 | Hot stage V5 |", "|---|---|---:|---:|---:|---:|---|"]
for row in SPEEDUPS:
    key = row["site"], row["size"]
    a, b = AVERAGES[3][key], AVERAGES[4][key]
    hot = max(range(1, 7), key=lambda i: b[f"S{i}_ms"])
    lines.append(f"| {key[0]} | {key[1]} | {a['S6_ms']:.6f} | {b['S6_ms']:.6f} | "
                 f"{a['S6_ms']/b['S6_ms']:.3f} | {100*b['S6_ms']/b['Total_ms']:.2f}% | S{hot} |")
lines += ["", "### Độ biến thiên của 5 runs", ""]
for row in VARIABILITY:
    if row["version"] == 4:
        lines.append(f"- {row['site']}/{row['size']}: CV={row['cv_pct']:.2f}%; runs_ms={row['runs_ms']}")
lines += ["", "### Trạng thái profiling", ""]
for row in PROFILE_RESULTS:
    lines.append(f"- {row['tool']} {row['site']}/{row['size']} {row.get('kernel', '')}: {'OK' if row['ok'] else 'THIẾU/LỖI'}")
lines += ["", "## 5. Analysis", "",
          "TODO: Trích số liệu NSYS/NCU kèm đường dẫn report và ảnh. Giải thích thay đổi occupancy, memory/compute throughput, warp stalls và launch gaps.", "",
          "TODO: Giải thích các case CV cao; đối chiếu S5 với S6 ở site2/large; xác định bottleneck còn lại theo bằng chứng.", "",
          "TODO: Kiểm tra gallery challenge, giữ trạng thái thất bại và không suy diễn nguyên nhân chỉ từ mã lỗi.", "",
          "TODO: Bổ sung link bằng chứng correctness P10-T02 và kết luận keep/revert.", ""]
report = "\n".join(lines)
(PACKAGE / "report/p10-t05-v5-apod-draft.md").write_text(report, encoding="utf-8")
log = ["# APOD V5 — bản nháp cập nhật", "",
       "| Site | Size | V5 Total ms | V5/V0 cùng phiên | V5/V3 cùng phiên | Kết luận |", "|---|---|---:|---:|---:|---|"]
for row in SPEEDUPS:
    cpu = f"{row['speedup_vs_v0']:.3f}" if row['speedup_vs_v0'] != "" else "thiếu baseline"
    log.append(f"| {row['site']} | {row['size']} | {row['Total_ms']:.6f} | {cpu} | {row['speedup_vs_prev']:.3f} | chờ phân tích |")
(PACKAGE / "report/apod-v5-update-draft.md").write_text("\n".join(log) + "\n", encoding="utf-8")
profile_ok = (len(PROFILE_RESULTS) == len(PROFILE_CASES) * sum(1 + len(v) for v in NCU_KERNELS.values())
              and all(r["ok"] for r in PROFILE_RESULTS))
screenshots = [p for p in (PROFILE / "screenshots").rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
STATUS = {"benchmark_validated": globals().get("BENCHMARK_OK", False),
          "same_session_v0_loaded": len(BASELINE) >= 9,
          "profiling_collected": profile_ok, "screenshot_files": len(screenshots),
          "analysis_reviewed": False, "uploaded_to_group_drive": False,
          "task_complete": False, "note": "Manual analysis, screenshots, source/dataset provenance review and handoff still required."}
dump(PACKAGE / "task_status.json", STATUS)
display(Markdown(report))
print(json.dumps(STATUS, indent=2, ensure_ascii=False))


## 6. Lưu kết quả và giữ runtime

Cell cuối đóng ZIP kèm checksum. Khi cấu hình Drive, ZIP được chép vào `v5/benchmark/packages/`; giải nén nội dung `v5/benchmark/` và `v5/profiling/` theo quy ước nhóm. Notebook không gọi disconnect, restart hay tạo runtime khác. Chỉ hoàn tất task khi đã phân tích profiling và lưu đủ bằng chứng.


In [ ]:
# @title Đóng gói và tải bằng chứng (kể cả khi profiling thất bại)
# Keep partial raw CSVs if benchmark validation or the subprocess failed.
partial = WORK / "benchmark_raw"
if partial.is_dir() and not globals().get("BENCHMARK_OK", False):
    shutil.copytree(partial, PACKAGE / "partial_benchmark", dirs_exist_ok=True)
    dump(PACKAGE / "task_status.json", {"task_complete": False, "benchmark_validated": False,
                                        "note": "Partial benchmark; inspect logs before using any measurements."})
if "source_snapshot" in globals() and SOURCE:
    try:
        current = source_snapshot()
        if current["snapshot_sha256"] != SOURCE["snapshot_sha256"]:
            dump(PACKAGE / "source_changed.json", current)
            print("SOURCE CHANGED: do not use these measurements without checking provenance.")
    except Exception as error:
        dump(PACKAGE / "source_check_error.json", {"error": str(error)})
        print("Source integrity check failed; retaining evidence for inspection:", error)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# Hash every artifact for later integrity checks; no dataset photos are packaged.
manifest = {p.relative_to(PACKAGE).as_posix(): digest(p) for p in sorted(PACKAGE.rglob("*"))
            if p.is_file() and p.name != "artifact_sha256.json"}
dump(PACKAGE / "artifact_sha256.json", manifest)
archive = Path(shutil.make_archive(str(WORK / f"Group16_P10_T05_Evidence_{stamp}"), "zip", PACKAGE))
checksum = digest(archive)
checksum_path = archive.with_suffix(".zip.sha256")
checksum_path.write_text(f"{checksum}  {archive.name}\n", encoding="utf-8")
print("Evidence:", archive, "bytes:", archive.stat().st_size, "SHA-256:", checksum)
if DRIVE_RESULTS_DIR.strip():
    destination = Path(DRIVE_RESULTS_DIR)
    if not MOUNT_DRIVE or not str(destination.resolve()).startswith("/content/drive/") or not destination.is_dir():
        raise ValueError("Mount Drive and provide an existing Drive results folder.")
    target = destination / "v5" / "benchmark" / "packages" / WORK.name / archive.name
    target.parent.mkdir(parents=True, exist_ok=False)
    shutil.copy2(archive, target)
    shutil.copy2(checksum_path, target.with_suffix(".zip.sha256"))
    print("Copied archive to Drive:", target)
files.download(str(archive))
print("Review task_status.json and finish the manual analysis before marking P10-T05 complete.")


Tài liệu: `docs/task-board.md`, `docs/results-guide.md`, `docs/report/p10-v5-verification.md`. Notebook gọi script và build trực tiếp từ `src/`; source manifest, hash binary và hash dataset được giữ trong gói bằng chứng.
